# 06 Predict

다음 회차의 후보 보상 등장 여부를 과거 패턴만으로 예측하는 누수 없는 실험 노트북이다. maplessunday의 블랙박스 예측과 달리 여기서는 재현 가능한 후보 행렬, 시간순 분할, 베이스라인, 모델 2종, 다음 회차 추론 결과를 한 흐름에서 비교한다.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.predict_dataset import DEFAULT_RECENCY_GATE_DAYS, DOMAIN_DISCONTINUED, build_inference_frame, write_prediction_frame

RANDOM_SEED = 42
SPLIT_CUTOFF = pd.Timestamp("2025-01-01")

## 데이터 로드

`data/processed/tidy_rewards.csv`와 `data/interim/predict_dataset.csv`만 사용한다. `data/raw`는 읽거나 수정하지 않는다.

후보 상위 N개는 train 구간(`< 2025-01-01`) 빈도로만 선정한다. 테스트 구간 정보가 후보 universe 설계에 들어가지 않도록 같은 컷오프를 후보 선정과 시간순 분할에 함께 사용한다.

In [ ]:
predict_path = ROOT / "data" / "interim" / "predict_dataset.csv"
tidy_path = ROOT / "data" / "processed" / "tidy_rewards.csv"

write_prediction_frame(
    tidy_path=tidy_path,
    output_path=predict_path,
    candidate_cutoff=SPLIT_CUTOFF,
)

df = pd.read_csv(predict_path, parse_dates=["date"])
summary = {
    "행": len(df),
    "회차": df["date"].nunique(),
    "후보": df["reward_norm"].nunique(),
    "양성비율": df["y"].mean(),
    "시작일": df["date"].min().date(),
    "종료일": df["date"].max().date(),
    "후보컷오프": SPLIT_CUTOFF.date(),
}
summary

## 시간순 분할

랜덤 분할은 사용하지 않는다. 기본 컷오프는 `2025-01-01`이며, 학습 구간의 마지막 날짜가 테스트 구간의 첫 날짜보다 앞서는지 확인한다.

In [ ]:
cutoff = SPLIT_CUTOFF
train = df.loc[df["date"] < cutoff].copy()
test = df.loc[df["date"] >= cutoff].copy()

split_summary = {
    "학습행": len(train),
    "테스트행": len(test),
    "학습시작일": train["date"].min().date(),
    "학습종료일": train["date"].max().date(),
    "테스트시작일": test["date"].min().date(),
    "테스트종료일": test["date"].max().date(),
    "학습양성비율": train["y"].mean(),
    "테스트양성비율": test["y"].mean(),
    "분할검증": train["date"].max() < test["date"].min(),
}
assert train["date"].max() < test["date"].min()
split_summary

## 평가 함수

점수는 확률 또는 확률에 준하는 연속값을 받는다. 회차별 적중률은 각 회차의 실제 양성 개수만큼 상위 후보를 골랐을 때 하나 이상 실제 양성을 포함한 회차 비율이다.

점수가 동점이면 `nlargest`가 상위 k개를 고르며, 이 값은 베이스라인 비교를 위한 근사 지표로만 해석한다.

In [ ]:
def evaluate(y_true, y_score, dates):
    y_true = pd.Series(y_true).astype(int).reset_index(drop=True)
    y_score = pd.Series(y_score).astype(float).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    y_pred = (y_score >= 0.5).astype(int)
    work = pd.DataFrame({"date": dates, "y": y_true, "score": y_score})

    def date_hit(group):
        k = int(group["y"].sum())
        if k < 1:
            return np.nan
        return float(group.nlargest(k, "score")["y"].max())

    auc = roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": auc,
        "date_hit_rate": work.groupby("date", sort=True).apply(date_hit).mean(),
    }

## 베이스라인

B1은 `days_since_last >= roll_gap_median` 규칙이다. B2는 과거 출현 비율 `past_rate`를 사전확률 점수로 사용한다. 이벤트 지표, 중단 여부, 미래 파생 변수는 사용하지 않는다.

`category`는 보상의 정적 속성으로 보고 전체 이력의 최빈값을 사용하며, 예측 시점에 알려진 값으로 간주한다.

In [ ]:
b1_score = (
    test["roll_gap_median"].notna()
    & (test["days_since_last"] >= test["roll_gap_median"])
).astype(float)
b2_score = test["past_rate"].clip(0, 1)

baseline_results = pd.DataFrame(
    [
        {"기준선": "간격규칙", **evaluate(test["y"], b1_score, test["date"])},
        {"기준선": "사전확률", **evaluate(test["y"], b2_score, test["date"])},
    ]
)
baseline_results

## 모델 입력 준비

`X_train`, `y_train`, `X_test`, `y_test`가 준비되어 있다. 로지스틱 회귀나 트리 계열 모델을 직접 작성하고 `predict_proba` 또는 이에 준하는 점수를 `evaluate()`로 베이스라인과 비교한다.

In [ ]:
feature_columns = [
    "days_since_last",
    "past_count",
    "past_rate",
    "roll_gap_mean",
    "roll_gap_median",
    "lag1_appeared",
    "category",
    "month",
    "quarter",
    "weekofyear",
]

X_all = pd.get_dummies(df[feature_columns], columns=["category"], dummy_na=True)
X_all = X_all.fillna(-1)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]
y_train = train["y"].astype(int)
y_test = test["y"].astype(int)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

## 모델 학습과 시간순 백테스트

모델은 로지스틱 회귀와 랜덤포레스트 2종만 사용한다. 두 모델 모두 `random_state=42`로 고정하고, 학습은 `2025-01-01` 이전 행만 사용한다. 평가는 베이스라인과 같은 `evaluate()` 함수로 테스트 구간 `2025-01-05`부터 `2026-06-07`까지 비교한다.

추론표는 발표 관점의 지표인 회차별 적중률(`date_hit_rate`)이 가장 높은 랜덤포레스트로 산출한다. 로지스틱 회귀는 `roc_auc`가 약간 높지만, 발표에서 필요한 비교 단위는 개별 후보 전체 순위보다 회차별 상위 후보가 실제 보상을 하나 이상 포함하는지에 더 가깝다.

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
tree_model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED)

logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

logistic_score = logistic_model.predict_proba(X_test)[:, 1]
tree_score = tree_model.predict_proba(X_test)[:, 1]

model_results = pd.DataFrame(
    [
        {"방법": "로지스틱", **evaluate(y_test, logistic_score, test["date"])},
        {"방법": "랜덤포레스트", **evaluate(y_test, tree_score, test["date"])},
    ]
)
comparison_results = pd.concat(
    [baseline_results.rename(columns={"기준선": "방법"}), model_results],
    ignore_index=True,
)
comparison_results

## 다음 회차 추론 보정

관측 가능한 마지막 원본 회차는 `2026-06-07`이다. `2026-06-14` 실제 보상은 인벤 원본이 아직 보상명을 채우지 않은 상태라 `data/raw`를 수정하지 않고, 사용자 제공 공식 확인값을 `data/interim/manual_obs_20260614.csv`에 별도 보조 관측으로 기록했다.

원시 모델은 오래 나오지 않은 단종 보상을 주기상 임박한 보상으로 오인할 수 있다. 그래서 전방 예보에는 기준일 미만 마지막 등장 후 180일을 넘긴 후보를 제외하는 재증 게이트와 도메인 단종 목록을 적용한다. 백테스트 비교표는 원시 모델 능력이고, 전방 예보는 재증·도메인 게이트를 적용한 결과다.

In [ ]:
manual_obs_path = ROOT / "data" / "interim" / "manual_obs_20260614.csv"
tidy_raw = pd.read_csv(tidy_path)
manual_obs = pd.read_csv(manual_obs_path, parse_dates=["date"])
category_lookup = tidy_raw.dropna(subset=["reward_norm", "category"]).groupby("reward_norm")["category"].agg(lambda s: s.mode().iloc[0])
manual_for_tidy = manual_obs.copy()
manual_for_tidy["category"] = manual_for_tidy["reward_norm"].map(category_lookup)
manual_for_tidy["is_future"] = False
manual_for_tidy["no_event_week"] = False
tidy_with_manual = pd.concat([tidy_raw, manual_for_tidy], ignore_index=True, sort=False)
actual_0614 = set(manual_obs["reward_norm"])
manual_summary = {
    "보조관측행": len(manual_obs),
    "보조관측일": manual_obs["date"].dt.date.unique().tolist(),
    "실제보상": sorted(actual_0614),
    "재증게이트일수": DEFAULT_RECENCY_GATE_DAYS,
    "도메인단종": sorted(DOMAIN_DISCONTINUED),
}
manual_summary

In [ ]:
def predict_for_date(tidy_source, target_date, gate=True):
    infer_frame = build_inference_frame(
        tidy_source,
        target_date,
        candidate_cutoff=SPLIT_CUTOFF,
        recency_gate_days=DEFAULT_RECENCY_GATE_DAYS if gate else None,
        domain_discontinued=DOMAIN_DISCONTINUED if gate else frozenset(),
    )
    X_target = pd.get_dummies(infer_frame[feature_columns], columns=["category"], dummy_na=True)
    X_target = X_target.reindex(columns=X_all.columns, fill_value=0)
    X_target = X_target.fillna(-1)
    scores = tree_model.predict_proba(X_target)[:, 1]
    table = infer_frame[["date", "reward_norm"]].copy()
    table["적중확률"] = scores
    table = table.sort_values(["적중확률", "reward_norm"], ascending=[False, True]).reset_index(drop=True)
    table["순위"] = np.arange(1, len(table) + 1)
    table["적중확률(%)"] = (table["적중확률"] * 100).round(1)
    return table[["순위", "date", "reward_norm", "적중확률", "적중확률(%)"]]

def verify_prediction(table, actual_rewards):
    work = table.copy()
    work["실제등장"] = work["reward_norm"].isin(actual_rewards)
    top5_hits = int(work.head(5)["실제등장"].sum())
    rows = []
    for reward in sorted(actual_rewards):
        matched = work.loc[work["reward_norm"] == reward]
        if matched.empty:
            rows.append({"reward_norm": reward, "순위": pd.NA, "적중확률(%)": pd.NA, "후보포함": False})
        else:
            row = matched.iloc[0]
            rows.append({"reward_norm": reward, "순위": int(row["순위"]), "적중확률(%)": float(row["적중확률(%)"]), "후보포함": True})
    return top5_hits, pd.DataFrame(rows)

## 2026-06-14 검증

`2026-06-14`는 이미 지나간 회차이므로 게이트 전 원시 추론과 게이트 후 추론을 실제 보조 관측과 대조한다. 선물류는 장기·특수 보상이라 후보권 밖에 있을 수 있으며, 그 경우도 실패로 그대로 기록한다.

In [ ]:
target_0614 = pd.Timestamp("2026-06-14")
pred_0614_raw = predict_for_date(tidy_raw, target_0614, gate=False)
pred_0614_gate = predict_for_date(tidy_raw, target_0614, gate=True)
raw_hits, raw_actual = verify_prediction(pred_0614_raw, actual_0614)
gate_hits, gate_actual = verify_prediction(pred_0614_gate, actual_0614)
verification_0614 = pd.DataFrame(
    [
        {"구분": "게이트 전", "상위5_적중수": raw_hits, "후보수": len(pred_0614_raw)},
        {"구분": "게이트 후", "상위5_적중수": gate_hits, "후보수": len(pred_0614_gate)},
    ]
)
raw_actual_labeled = raw_actual.assign(**{"구분": "게이트 전"})
gate_actual_labeled = gate_actual.assign(**{"구분": "게이트 후"})
actual_rank_0614 = pd.concat(
    [raw_actual_labeled, gate_actual_labeled],
    ignore_index=True,
)[["구분", "reward_norm", "후보포함", "순위", "적중확률(%)"]]
display(verification_0614)
display(actual_rank_0614)
display(pred_0614_raw.head(5)[["순위", "reward_norm", "적중확률(%)"]])
display(pred_0614_gate.head(5)[["순위", "reward_norm", "적중확률(%)"]])

## 2026-06-21 예보

`2026-06-21` 예보는 `2026-06-14` 보조 관측을 기준일 이전 이력으로 포함한다. 따라서 06-14에 실제 등장한 몬스터파크 보상은 직전 등장 직후로 처리되어 재등장 점수가 낮아지는 것이 정상 동작이다. 아래 확률은 원 모델 점수의 백분율 표기이며 정규화하거나 재배분하지 않는다.

In [ ]:
target_0621 = pd.Timestamp("2026-06-21")
forecast_0621 = predict_for_date(tidy_with_manual, target_0621, gate=True)
forecast_0621_top5 = forecast_0621.head(5)[["순위", "reward_norm", "적중확률(%)"]].rename(columns={"reward_norm": "보상"})
excluded_check_0621 = {
    "현상금_경험치": "현상금_경험치" not in set(forecast_0621["reward_norm"]),
    "불꽃늑대_경험치": "불꽃늑대_경험치" not in set(forecast_0621["reward_norm"]),
    "스타포스_성공확률": "스타포스_성공확률" not in set(forecast_0621["reward_norm"]),
    "스타포스_1+1강화": "스타포스_1+1강화" not in set(forecast_0621["reward_norm"]),
}
assert all(excluded_check_0621.values())
display(forecast_0621_top5)
excluded_check_0621

## 결론 프레임

이 노트북의 예측은 다음 회차 후보를 과거 출현 패턴으로 정렬하는 보조 지표다. 예측 신호는 단순 빈도가 아니라 주기 위상이다. 최빈 보상이라도 직전 회차에 나왔다면 낮게 점수화될 수 있고, 특정 주의 성장보다 재등장 주기가 맞물린 보상이 높게 잡힐 수 있다.

06-14 검증은 원시 모델이 단종·휴면 보상을 과대평가할 수 있음을 보여 준다. 재증 게이트와 도메인 단종 목록은 이 문제를 줄이기 위한 전방 예보용 필터이며, 선물류처럼 과거 주기로 설명하기 어려운 특수 보상까지 맞힌다는 뜻은 아니다.

maplessunday와의 차이는 성능 우위 주장이 아니라 투명성과 재현성이다. 같은 입력 데이터, 같은 시간순 분할, 같은 후보 universe에서 산출 과정을 다시 실행할 수 있다는 점을 프로젝트의 차별점으로 둔다.